In [ ]:
import pandas as pd
import numpy as np
import re
import pickle

import nltk
from nltk.corpus import stopwords



from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

In [ ]:
df = pd.read_csv('data.csv')
df.head()

,Reviewer Name,Review Title,Place of Review,Up Votes,Down Votes,Month,Review text,Ratings
0,Kamal Suresh,Nice product,"Certified Buyer, Chirakkal",889.0,64.0,Feb 2021,"Nice product, good quality, but price is now r...",4
1,Flipkart Customer,Don't waste your money,"Certified Buyer, Hyderabad",109.0,6.0,Feb 2021,They didn't supplied Yonex Mavis 350. Outside ...,1
2,A. S. Raja Srinivasan,Did not meet expectations,"Certified Buyer, Dharmapuri",42.0,3.0,Apr 2021,Worst product. Damaged shuttlecocks packed in ...,1
3,Suresh Narayanasamy,Fair,"Certified Buyer, Chennai",25.0,1.0,NaN,"Quite O. K. , but nowadays the quality of the...",3
4,ASHIK P A,Over priced,NaN,147.0,24.0,Apr 2016,Over pricedJust â?¹620 ..from retailer.I didn'...,1


In [ ]:
print("Dataset Shape:", df.shape)
print("Columns:\n", df.columns)
print("Sample Data:\n", df.head())

Dataset Shape: (8518, 8)
Columns:
 Index(['Reviewer Name', 'Review Title', 'Place of Review', 'Up Votes',
       'Down Votes', 'Month', 'Review text', 'Ratings'],
      dtype='object')
Sample Data:
             Reviewer Name               Review Title  \
0            Kamal Suresh               Nice product   
1       Flipkart Customer     Don't waste your money   
2  A. S. Raja Srinivasan   Did not meet expectations   
3     Suresh Narayanasamy                       Fair   
4               ASHIK P A                Over priced   

               Place of Review  Up Votes  Down Votes     Month  \
0   Certified Buyer, Chirakkal     889.0        64.0  Feb 2021   
1   Certified Buyer, Hyderabad     109.0         6.0  Feb 2021   
2  Certified Buyer, Dharmapuri      42.0         3.0  Apr 2021   
3     Certified Buyer, Chennai      25.0         1.0       NaN   
4                          NaN     147.0        24.0  Apr 2016   

                                         Review text  Ratings  
0  

In [ ]:
print("Rating Distribution:\n", df['Ratings'].value_counts())

Rating Distribution:
 Ratings
5    5080
4    1746
1     769
3     615
2     308
Name: count, dtype: int64


In [ ]:
# CREATE SENTIMENT LABELS


def create_sentiment(rating):

    if rating >= 4:
        return 1
    elif rating <= 2:   
        return 0
    else:
        return np.nan

df['sentiment'] = df['Ratings'].apply(create_sentiment)
df = df.dropna(subset=['sentiment'])


print("After Removing Neutral Reviews:", df.shape)

After Removing Neutral Reviews: (7903, 9)


In [ ]:
# Data Cleaning
stop_words = stopwords.words('english')
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z ]', '', text) # Remove special characters
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_review'] = df['Review text'].apply(clean_text)


print("Cleaned Text Sample:\n", df['clean_review'].head())

Cleaned Text Sample:
 0    nice product good quality price rising bad sig...
1    didnt supplied yonex mavis outside cover yonex...
2    worst product damaged shuttlecocks packed new ...
4    pricedjust retaileri didnt understand wat adva...
5              good quality product delivered timeread
Name: clean_review, dtype: object


In [ ]:
# TEXT EMBEDDING (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000)

x = tfidf.fit_transform(df['clean_review'])
y = df['sentiment']

print("TF-IDF Shape:", x.shape)

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, random_state=42, stratify=y
)

# model training
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model Training Completed")

# MODEL EVALUATION
y_pred = model.predict(X_test)

f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

print("Classification Report:\n")
print(classification_report(y_test, y_pred))

TF-IDF Shape: (7903, 3539)
Model Training Completed
F1 Score: 0.9544008483563097
Classification Report:

              precision    recall  f1-score   support

         0.0       0.86      0.47      0.61       215
         1.0       0.92      0.99      0.95      1366

    accuracy                           0.92      1581
   macro avg       0.89      0.73      0.78      1581
weighted avg       0.91      0.92      0.91      1581



In [ ]:
pickle.dump(model, open('sentiment_model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))


print("Model and Vectorizer Saved")

Model and Vectorizer Saved
